# Data Description

- Background: 對1,223位G5/G6缺值候選人的全國補查原始回應、完整擷取文字與人工判讀。採集開始11/08，判讀與本文件完成11/09；資料作者包括各公開來源，文件由RUEI-HSIANG, CHANG（B12303030；AI協助）整理。
- date received: 20261108–20261109（本次實際採集期間；不是各來源的出版日期）
- Path to data file: 各筆原始`query`／`url`／`source_url`，保存於下表精確本地檔案。
- Unit of observation: search_responses一列工具批次、內含最多4個查詢及完整回應；source_bodies一列URL擷取；reviewed_adjudications一列候選人×變數已決格；其他JSON為計畫、索引、來源線索與未決註記。
- Sample period: 2026候選人資料，來源跨歷史選舉／人物背景；採集11/08–11/09，未使用2026投票結果。
- Known issues: 共2,792式、699批，2,250來源中2,090筆保存文字、160次失敗。captured_not_yet_read是擷取當下狀態，實際已讀並決定的279格另存判讀表；未使用或未決來源不得據此填0。部分HTML選擇器只取到導航、其他縣市同名人或分頁第一頁，已決定者須另有可用全文。跨午夜部分batch沿用11/08常數，精確UTC時刻未保留；日期範圍據實揭露。
- Definition for each variable: query_plan欄為candidate_key／candidate_name／county_city／round／query，round1–2為G6、3為G5、4為補中文別名；search_responses另含batch_start／queries／response／retrieval_date／response_sha256。source_bodies含id／url／title／selector／text／body_sha256／http_status／capture_date／status／people（線索連結）或error；**hash僅確認保存文字身分**。adjudications含candidate_key／variable／value／來源id與URL／body_sha256／evidence_note／supporting_excerpt／quality／full_article_body_read／checked_date與可選supporting_files。其餘線索檔內careers／candidate_paragraphs為定位用，不是已核實編碼。
- Data path: 下表12個實際原始檔。
- Role: raw / audit evidence；未審線索與失敗擷取不是analysis-ready values。
- Verification status: 本文件clean kernel只讀驗證精確SHA、JSON／JSONL解析與筆數、查詢覆蓋、本文hash、279個判讀鍵與來源存在；不代表全1,223人完成判讀。
- SHA-256: 下表及程式固定常數。
- Next processing notebook: [data/code/restore_national_g5_g6_20261109.ipynb](../code/restore_national_g5_g6_20261109.ipynb)

| Exact data path | SHA-256 |
| --- | --- |
| `data/raw/national_gap_restore_20261108/G5_profile_source_leads.json` | `81d3288ae6a716b278cb91cd1f1b185619890dae8a8bae16995636a26c98dd26` |
| `data/raw/national_gap_restore_20261108/alias_query_plan.json` | `7c5f51949ab6381a115deb42920df66c394b195d611a18c8389b6f820108b8d3` |
| `data/raw/national_gap_restore_20261108/article_capture_manifest.json` | `e6b6f0d97b2fe6f92498acc4337bef2e6774ca8c043716f614978e13d3f80a2d` |
| `data/raw/national_gap_restore_20261108/collection_checkpoint.json` | `37ed1c2288eaaeebaad546f0e780c947dcf2020d205a7dbe85e5b0d75e8ddbe8` |
| `data/raw/national_gap_restore_20261108/county_election_articles.json` | `bfa870d191c9887d12e3c704904d95acd77b7341a5aa9fbb27e17c43aea6cd33` |
| `data/raw/national_gap_restore_20261108/followup_leads.json` | `e6a7565825c1f570f56d372266c8b7e69c5fb636a7dd0642dddfa1b8e9ef3e11` |
| `data/raw/national_gap_restore_20261108/g5_followup_query_plan.json` | `3e062f80e0698b99f82c2dda5b8e29f9367f3fe3b88e47ff5ef83ae765695956` |
| `data/raw/national_gap_restore_20261108/identity_and_scope_review_notes.json` | `fd3c26642cea8aa2f30c178122c2f9ecdf01a2a3f2d91332d1ade3682d1e1d16` |
| `data/raw/national_gap_restore_20261108/query_plan.json` | `81643739e1f014604227654f45071a3edb566fd353f453ef7130bcd06f515c5f` |
| `data/raw/national_gap_restore_20261108/reviewed_adjudications.jsonl` | `520a17aa37ffdeed6786dc1fffe62c118cc5a7872a09994b8ebe61a48620d445` |
| `data/raw/national_gap_restore_20261108/search_responses.jsonl` | `ac561219ed9c51fbf51c44cd79d63e41955ee75734b2e7d2bf6fdd49601e7e0f` |
| `data/raw/national_gap_restore_20261108/source_bodies.jsonl` | `b8908403ebe6ce4f5a970bcdc8a1bbde10ac0348cc96335bc800c965fac7ec0a` |


In [1]:
from pathlib import Path
import json, hashlib, re
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def jl(p):return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]


## Change for Your Own Data

只從下方逐檔常數讀入，不從檔案日期挑版本，不寫回原始資料。

In [2]:
RAW_PINS={'data/raw/national_gap_restore_20261108/G5_profile_source_leads.json': '81d3288ae6a716b278cb91cd1f1b185619890dae8a8bae16995636a26c98dd26', 'data/raw/national_gap_restore_20261108/alias_query_plan.json': '7c5f51949ab6381a115deb42920df66c394b195d611a18c8389b6f820108b8d3', 'data/raw/national_gap_restore_20261108/article_capture_manifest.json': 'e6b6f0d97b2fe6f92498acc4337bef2e6774ca8c043716f614978e13d3f80a2d', 'data/raw/national_gap_restore_20261108/collection_checkpoint.json': '37ed1c2288eaaeebaad546f0e780c947dcf2020d205a7dbe85e5b0d75e8ddbe8', 'data/raw/national_gap_restore_20261108/county_election_articles.json': 'bfa870d191c9887d12e3c704904d95acd77b7341a5aa9fbb27e17c43aea6cd33', 'data/raw/national_gap_restore_20261108/followup_leads.json': 'e6a7565825c1f570f56d372266c8b7e69c5fb636a7dd0642dddfa1b8e9ef3e11', 'data/raw/national_gap_restore_20261108/g5_followup_query_plan.json': '3e062f80e0698b99f82c2dda5b8e29f9367f3fe3b88e47ff5ef83ae765695956', 'data/raw/national_gap_restore_20261108/identity_and_scope_review_notes.json': 'fd3c26642cea8aa2f30c178122c2f9ecdf01a2a3f2d91332d1ade3682d1e1d16', 'data/raw/national_gap_restore_20261108/query_plan.json': '81643739e1f014604227654f45071a3edb566fd353f453ef7130bcd06f515c5f', 'data/raw/national_gap_restore_20261108/reviewed_adjudications.jsonl': '520a17aa37ffdeed6786dc1fffe62c118cc5a7872a09994b8ebe61a48620d445', 'data/raw/national_gap_restore_20261108/search_responses.jsonl': 'ac561219ed9c51fbf51c44cd79d63e41955ee75734b2e7d2bf6fdd49601e7e0f', 'data/raw/national_gap_restore_20261108/source_bodies.jsonl': 'b8908403ebe6ce4f5a970bcdc8a1bbde10ac0348cc96335bc800c965fac7ec0a'}
for rel,expected in RAW_PINS.items():assert sha(ROOT/rel)==expected,rel
objects={}
for rel in RAW_PINS:
    p=ROOT/rel;objects[rel]=jl(p) if p.suffix=='.jsonl' else json.loads(p.read_text())
summary=[]
for rel,obj in objects.items():summary.append({'path':rel,'structure':type(obj).__name__,'top_level_n':len(obj),'sha256':RAW_PINS[rel]})
display(pd.DataFrame(summary))
search=objects['data/raw/national_gap_restore_20261108/search_responses.jsonl']
articles=objects['data/raw/national_gap_restore_20261108/source_bodies.jsonl']
decisions=objects['data/raw/national_gap_restore_20261108/reviewed_adjudications.jsonl']
assert len(search)==699 and sum(len(b['queries']) for b in search)==2792
for b in search:assert hashlib.sha256(b['response'].encode()).hexdigest()==b['response_sha256']
for a in articles:
    if 'text' in a:assert hashlib.sha256(a['text'].encode()).hexdigest()==a['body_sha256']
assert len({a['id'] for a in articles})==len(articles)==2250
assert len({(x['candidate_key'],x['variable']) for x in decisions})==len(decisions)==279
print('PASS: raw files parsed and hashes checked. Captures are not automatic coding decisions.')


,path,structure,top_level_n,sha256
0,data/raw/national_gap_restore_20261108/G5_prof...,list,140,81d3288ae6a716b278cb91cd1f1b185619890dae8a8bae...
1,data/raw/national_gap_restore_20261108/alias_q...,list,4,7c5f51949ab6381a115deb42920df66c394b195d611a18...
2,data/raw/national_gap_restore_20261108/article...,list,2250,e6b6f0d97b2fe6f92498acc4337bef2e6774ca8c043716...
3,data/raw/national_gap_restore_20261108/collect...,dict,15,37ed1c2288eaaeebaad546f0e780c947dcf2020d205a7d...
4,data/raw/national_gap_restore_20261108/county_...,list,20,bfa870d191c9887d12e3c704904d95acd77b7341a5aa9f...
5,data/raw/national_gap_restore_20261108/followu...,list,417,e6a7565825c1f570f56d372266c8b7e69c5fb636a7dd06...
6,data/raw/national_gap_restore_20261108/g5_foll...,list,346,3e062f80e0698b99f82c2dda5b8e29f9367f3fe3b88e47...
7,data/raw/national_gap_restore_20261108/identit...,list,9,fd3c26642cea8aa2f30c178122c2f9ecdf01a2a3f2d913...
8,data/raw/national_gap_restore_20261108/query_p...,list,2442,81643739e1f014604227654f45071a3edb566fd353f453...
9,data/raw/national_gap_restore_20261108/reviewe...,list,279,520a17aa37ffdeed6786dc1fffe62c118cc5a7872a0999...


PASS: raw files parsed and hashes checked. Captures are not automatic coding decisions.


## Summary

### Sample

In [3]:
display(pd.DataFrame(decisions).drop(columns=['supporting_files'],errors='ignore').head(10))
display(pd.DataFrame([{'batch_start':b['batch_start'],'query_n':len(b['queries']),'date':b['retrieval_date'],'response_chars':len(b['response'])} for b in search]).head(10))


,candidate_key,candidate_name,county_city,variable,value,article_id,source_url,body_sha256,evidence_note,supporting_excerpt,quality,full_article_body_read,checked_date
0,2026|直轄市議員|新北市第10選舉區|林裔綺,林裔綺,新北市,G6_political_family,1,d571db4f40d175bc2561,https://zh.wikipedia.org/wiki/%E6%9E%97%E8%A3%...,bf2226dd12ce779fcde388450430b227d70260f8195e23...,配偶顏世雄曾任新北市議員／瑞芳鎮長；不採頁面尚未投票的2026得票表。,前新北市議員顏世雄之妻。,full_article_body,True,2026-11-09
1,2026|直轄市議員|新北市第11選舉區|白珮茹,白珮茹,新北市,G6_political_family,1,ce048b90d6d48fb136b2,https://news.tvbs.com.tw/local/2623203,d32a2fe3c7ab5a9d17a2f70ee6d65d0298ba5c2d44871f...,父親白添枝為前立委，一親等；僅編親屬／任職，不把報導涉案內容當本次分析變數。,白珮茹父親，也就是前立委白添枝,full_article_body,True,2026-11-09
2,2026|直轄市議員|新北市第1選舉區|鄭宇恩,鄭宇恩,新北市,G6_political_family,1,5753d66ba4a7fb1ee59f,https://tw.news.yahoo.com/-123459270.html,c101cb1c530e2c5e243a2ac54318ace16debc8e17bd433...,母親曾任淡水市民代表，一親等；父親單純黨職不作正碼依據。,母親曾任淡水區二屆市代表,full_article_body,True,2026-11-09
3,2026|直轄市議員|新北市第1選舉區|陳家琪,陳家琪,新北市,G6_political_family,1,a704c0aa909ae01d8d17,https://tw.news.yahoo.com/%E6%96%B0%E5%8C%97-%...,a7ea114bd6e5cb4368df40795c0c5274583ddab4f42115...,公公楊秀雄為八里前代表會主席，屬範圍內直系姻親；父親黨職不單獨作依據。,公公楊秀雄，是八里前代表會主席,full_article_body,True,2026-11-09
4,2026|直轄市議員|新北市第2選舉區|戴翎育,戴翎育,新北市,G6_political_family,1,300f28ffb18d146a5216,https://reader.turnnewsapp.com/ctl/20260118/B1...,d4ce6ad9b1488eeba678f38a1bca87ab3bcb908350f160...,母親賴秋媚曾任新北市議員，一親等。,前市議員賴秋媚的女兒戴翎育,full_article_body,True,2026-11-09
5,2026|直轄市議員|新北市第2選舉區|蔡佩呈,蔡佩呈,新北市,G6_political_family,1,300f28ffb18d146a5216,https://reader.turnnewsapp.com/ctl/20260118/B1...,d4ce6ad9b1488eeba678f38a1bca87ab3bcb908350f160...,父親宋明宗為現任新北市議員，一親等。,國民黨現任議員宋明宗傾向由女兒蔡佩呈接手參選,full_article_body,True,2026-11-09
6,2026|直轄市議員|新北市第2選舉區|鄭丞宏,鄭丞宏,新北市,G6_political_family,1,300f28ffb18d146a5216,https://reader.turnnewsapp.com/ctl/20260118/B1...,d4ce6ad9b1488eeba678f38a1bca87ab3bcb908350f160...,父親鄭漢騰曾任五股鄉長，一親等。,前五股鄉長鄭漢騰之子,full_article_body,True,2026-11-09
7,2026|直轄市議員|新北市第11選舉區|何元楷,何元楷,新北市,G6_political_family,0,8b170c870f2170688117,https://teamho666.oen.tw/,fda9939e5f2c6c9a2f78a9d16d6f2d05b2c6dd6f078cca...,候選人官網全文述父親為皮鞋工人、本人為幕僚；兩輪指定搜尋未發現符合公職親屬。0僅搜尋未發現。,,candidate_self_description_and_search_nonfinding,True,2026-11-09
8,2026|直轄市議員|新北市第11選舉區|劉財鑫,劉財鑫,新北市,G6_political_family,0,6e3a0902262ac62616a6,https://xn--2lw665d.tw/politician/d67a3894-c88...,c18fa943d021cfc344b3c5ea455451426b2a991102c3ec...,兩輪指定搜尋及完整候選人資料頁未發現符合公職親屬；頁面本身標尚未查核且無家族資訊，0為使用者...,,limited_web_visibility_search_nonfinding,True,2026-11-09
9,2026|直轄市議員|新北市第11選舉區|郭書成,郭書成,新北市,G6_political_family,0,b506a589bee42875a601,https://news.nsysu.edu.tw/p/404-1120-231532.ph...,d3805ecda34020787a25c68b8a193b801c07a18cffdbf1...,校方完整人物報導及兩輪指定搜尋未發現符合公職親屬；其他政治人物的政二代對比不可錯置給本人。,,full_profile_and_search_nonfinding,True,2026-11-09


,batch_start,query_n,date,response_chars
0,0,4,2026-11-08,10267
1,4,4,2026-11-08,10255
2,8,4,2026-11-08,9951
3,12,4,2026-11-08,9796
4,16,4,2026-11-08,10545
5,20,4,2026-11-08,9202
6,24,4,2026-11-08,10229
7,28,4,2026-11-08,8984
8,32,4,2026-11-08,9601
9,36,4,2026-11-08,10508


### Summary Stats

In [4]:
a=pd.DataFrame(decisions);display(a.groupby(['variable','value','quality']).size().rename('cells').reset_index())
print('Source text count:',sum('text' in x for x in articles),'failed:',sum('text' not in x for x in articles))
print('Reviewed cells:',len(a),'distinct candidates:',a.candidate_key.nunique())


,variable,value,quality,cells
0,G5_first_time_any_elected_office,0,current_candidate_prior_elected_office_full_ar...,22
1,G5_first_time_any_elected_office,0,current_candidate_prior_election_full_source,18
2,G5_first_time_any_elected_office,0,current_candidate_prior_election_full_text,51
3,G5_first_time_any_elected_office,0,prior_election_current_MOI_career_identity_sup...,1
4,G5_first_time_any_elected_office,1,explicit_first_run_with_identity_anchors,2
5,G5_first_time_any_elected_office,1,yang_newcomer_full_article_identity_anchors,1
6,G6_political_family,0,candidate_self_description_and_search_nonfinding,1
7,G6_political_family,0,full_article_and_search_nonfinding,2
8,G6_political_family,0,full_article_subject_checked_search_nonfinding,1
9,G6_political_family,0,full_biography_and_alias_search_nonfinding,1


Source text count: 2090 failed: 160
Reviewed cells: 279 distinct candidates: 243
